## Processing Tool for Swissleague HF Overal Ranking

The needed Imports:


In [1]:
import pandas as pd
import json
from pathlib import Path
!pip install reportlab
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Flowable, Paragraph, Spacer
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import Image
from reportlab.lib.utils import ImageReader
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.colors import grey
from collections import defaultdict

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.5 MB/s eta 0:00:00


Add Drive to the Notebook to access data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Add path to excel files

In [3]:
drive_path = Path('/content/drive')
files_path = drive_path / 'MyDrive/Swissleague'

Run this cell only at the begining of the season to generate the List of Competions that count to the Swisscup. This will generate a new file and completly overwrites the old one!

In [5]:
#########################################
# Set here the data:

competition_titles = ['Jura HF (SM 2026)', 'Jura Airtour', 'Engelberg Cup', 'Gantrisch HF', 'Eigertour', 'Gruyère Fly', 'HF Lungern', 'Flyback Frutigen', 'Trailfly', 'Vercofly' 'Beizenfliegen', 'Belli in Fly', 'Millets Cup']
competition_keys = ['jhf', 'airtour', 'engelberg', 'ghf', 'eiger', 'gruyere', 'lungern', 'flyback', 'trailfly', 'vercofly', 'beizen', 'belli', 'millet']
phsical = [False, False, False, False, False, False, False, False, False, False, False, True, True]
competition_file_name = "swissleague_competitions_2026.json"

#########################################
competition_path = files_path / competition_file_name
competition_json = {}

for title, key, physical in zip(competition_titles, competition_keys, phsical):
    competition_json[key] = {
        "title": title,
        "num_participants": 0,
        "physical": physical
    }

''' with open(competition_path, "w") as f:
    json.dump(competition_json, f, indent=4) '''

' with open(competition_path, "w") as f:\n    json.dump(competition_json, f, indent=4) '

Specify Titels and Paths to Data

In [6]:
# Output Titles
overall_title = "Swissleague Hike and Fly Overall Ranking"
women_title = "Swissleague Hike and Fly Female Ranking"
men_title = "Swissleague Hike and Fly Male Ranking"

# Datapath to the swissleague logo for the PDF
logo_path = files_path / "swisscup_hf_farbe.png"

# Kleingedrucktes
info = "For feedbacks, please contact: sport@shv-fsvl.ch"
info_not_enough_participants = "(*) Bei weniger als 5 Teilnehmer:innen erfolgt keine Wertung."

# Datapath to preprocessed JSON
json_path = files_path / "swissleague_data_2026.json"
competition_path = files_path / "swissleague_competitions_2026.json"

# Datapaths to competions excel
jura_path = "jhf_2025.xlsx"
engelberg_path = "EngelbergCup_2025.xlsx"
airtour_path = "airtour_2025_2.xlsx"
ghf_path = "ghf_2025.xlsx"
eiger_path = "eigertour_2025.xlsx"
sm_path = "SM_2025.xlsx"
flyback_path = "Flyback_2025.xlsx"
trailfly_path = "trailfly_2025.xlsx"
vercofly_path = "Vercofly_2025.xlsx"
beizen_path = "Beizen_2025.xlsx"
belli_path = "Belli_2025.xlsx"
millet_path = "Millets_2025.xlsx"

# specify which Excel files shall be processed and corresponding json keys
excel_files = [jura_path, engelberg_path, airtour_path, ghf_path, eiger_path,
               sm_path, flyback_path, trailfly_path, vercofly_path, beizen_path,
               belli_path, millet_path]
json_keys = [ "jhf", "engelberg", "airtour", "ghf", "eiger", "sm",
    "flyback", "trailfly", "vercofly", "beizen", "belli", "millet"]

Load the json file

In [7]:
json_data = {}
# check if json already exists:
try:
    with open(json_path, "r") as f:
        json_data = json.load(f)
except FileNotFoundError:
    print("Eval JSON file does not exist yet.")
    json_data = {}

competition_json = {}
try:
    with open(competition_path, "r") as f:
        competition_json = json.load(f)
except FileNotFoundError:
    print("Competition JSON file does not exist yet.")
    competition_json = {}

Functions used later in the script

In [ ]:
def print_name_discrepancy(json_data, row, key):
      print(f"\nDiscrepancy found for civl_id {row['civl_id']}:")
      print(f"{key} in JSON: {json_data[str(row['civl_id'])][key]}, {key} in Excel: {row[key]}")

def check_civl_id_name_discrepancy(json_data, row):
      if json_data[str(row['civl_id'])]['name'] != row['name']:
        print_name_discrepancy(json_data, row, 'name')
      if json_data[str(row['civl_id'])]['first_name'] != row['first_name']:
        print_name_discrepancy(json_data, row, 'first_name')

def check_wing(json_data, row):
      if json_data[str(row['civl_id'])]['glider'] != row['glider']:
        if json_data[str(row['civl_id'])]['glider'] == "":
          print(f"\nUpdating empty glider for {row['civl_id']}")
          json_data[str(row['civl_id'])]['glider'] = row['glider']
        else:
          print(f"\nDiscrepancy in glider found for civl_id {row['civl_id']}:")
          print("old glider: ", json_data[str(row['civl_id'])]['glider'])
          print("new glider: ", row['glider'])

def add_points_to_data(df: pd.DataFrame, num_participants):
    # Formula: 100 - 100 * (rank - 1) / (num_participants -1)
    df['points'] = (100 - (100 * (df['rank'] - 1)) / (num_participants - 1)).clip(lower=1).round(2)

    return num_participants, df

def check_if_competion_already_exists(json_data, civil_id, competition_key):
    if competition_key in json_data[civil_id]["competitions"]:
      print(f"Competition {competition_key} already exists for civl_id {civil_id}")
      return True
    return False

def add_competition_data(json_data, data_row, competition_key):
    json_data[str(data_row['civl_id'])]["competitions"][competition_key] = {
        "rank": data_row['rank'],
        "points": data_row['points'],
        "counts": True
    }
    return json_data

def update_total_points(json_data, civil_id:str, new_points, new_comp_key):
    athletes_comp_keys = json_data[civil_id]["competitions"].keys()
    if len(athletes_comp_keys) <= 4:
        total_points = json_data[civil_id]["total_points"] + new_points
        json_data[civil_id]["total_points"] = round(total_points, 2)

    else:
        # Find lowest points from previous comps
        lowest_points = new_points
        lowest_key = new_comp_key

        for comp_key in athletes_comp_keys:
            if json_data[civil_id]["competitions"][comp_key]['counts'] and json_data[civil_id]["competitions"][comp_key]['points'] < lowest_points:
              lowest_points = json_data[civil_id]["competitions"][comp_key]['points']
              lowest_key = comp_key

        # update remove lowest_competition from total points
        json_data[civil_id]["competitions"][lowest_key]['counts'] = False
        json_data[civil_id]["total_points"] += new_points - lowest_points
        json_data[civil_id]["total_points"] = round(json_data[civil_id]["total_points"], 2)
    return json_data

def remove_athletes_data_from_comp_keys(athletes_comp_keys):
    exclude_keys = {'name', 'first_name', 'gender', 'total_points', 'nat', 'glider'}
    return [key for key in athletes_comp_keys if key not in exclude_keys]

def add_athlete_data(json_data, data_row):
    print()
    print(f"Adding athlete data for civl_id {data_row['civl_id']}")
    json_data[str(data_row['civl_id'])] = {
        "name": data_row['name'],
        "first_name": data_row['first_name'],
        "gender": data_row['gender'],
        "total_points": 0,
        "nat": data_row['nat'],
        "glider": data_row['glider'],
        "competitions": {}
    }
    return json_data

def add_data_to_json(json_data: dict, df: pd.DataFrame, competition_key: str):
    # add data to json
    for _, row in df.iterrows():
        # If fake civl_id, check if pilot already exists in database
        if "Z" in str(row['civl_id']):
            # Search in JSON for same last name
            for json_civl_id, athlete in json_data.items():
                if athlete['name'] == row['name']:
                    # Check if first name matches
                    if athlete['first_name'] == row['first_name']:
                        # Replace manual civl_id with real one
                        row['civl_id'] = json_civl_id
                        break

        # check data if civl_id already exists
        if str(row['civl_id']) in json_data:
            # check if there is name discrepancy for manual update
            check_civl_id_name_discrepancy(json_data, row)
            check_wing(json_data, row)

        else:
            # add athletes data
            json_data = add_athlete_data(json_data, row)

        # add the competion
        if not check_if_competion_already_exists(json_data, str(row['civl_id']), competition_key):
          json_data = add_competition_data(json_data, row, competition_key)
          json_data = update_total_points(json_data, str(row['civl_id']), row['points'], competition_key)

    return json_data

def get_highes_fake_civil_id_already_in_use(json_data):
    # Extract all Fake ID's in use
    existing_fake_ids = [cid for cid in json_data.keys() if cid.startswith("ZZ")]

    if existing_fake_ids:
        max_num = max(int(cid[2:]) for cid in existing_fake_ids)  # take numeric part after "ZZ"
    else:
        max_num = 0  # start fresh if no fake IDs exist

    # New Fake ID must be one number higher
    fake_id_counter = max_num + 1

    return fake_id_counter

Now load the excel files and add the data to the JSON

In [ ]:
for competition_key, excel_file in zip(json_keys, excel_files):
    # Load the Excel file
    excel_path = f"{files_path}/Resultate/{excel_file}"
    print(f"Processing {excel_path}")
    df = pd.read_excel(excel_path)

    # Select only the necessary columns and rename for consistency
    df = df[['Rank', 'First Name', 'Last Name', 'Gender', 'CIVL ID', 'Nat', 'Glider']]
    df.columns = ['rank', 'first_name', 'name', 'gender', 'civl_id', 'nat', 'glider']

    # Evaluting the Points received based on ranking
    num_participants = competition_json[competition_key]["num_participants"]
    num_part, df = add_points_to_data(df, num_participants)

    # removing possible blanc spaces around civil id
    df['civl_id'] = df['civl_id'].astype(str).str.strip()

    fake_id_counter = get_highes_fake_civil_id_already_in_use(json_data)

    def replace_zero_with_fake(cid):
        global fake_id_counter
        if cid == "0":
            new_cid = f"ZZ{fake_id_counter:03d}"  # e.g. ZZ001, ZZ002
            fake_id_counter += 1
            return new_cid
        return cid

    # Replace civil id's "0" with a fake ID
    df['civl_id'] = df['civl_id'].apply(replace_zero_with_fake)

    # remove trailing blank spaces
    df['name'] = df['name'].str.strip().str.title()
    df['first_name'] = df['first_name'].str.strip().str.title()
    df['glider'] = df['glider'].fillna("").str.strip()

    # add data to json
    json_data = add_data_to_json(json_data, df, competition_key)

Processing /content/drive/MyDrive/Swissleague/Resultate/jhf_2025.xlsx

Discrepancy in glider found for civl_id 1414:
old glider:  Advance Omega ULS
new glider:  Advance Omega

Discrepancy in glider found for civl_id 85387:
old glider:  Ozone Enzo 3
new glider:  Ozone Enzo3

Discrepancy in glider found for civl_id 35556:
old glider:  Ozone Zeolite 2
new glider:  Ozone Enzo3

Discrepancy in glider found for civl_id 87138:
old glider:  Niviuk Artik 7 P
new glider:  Niviuk artik 7p

Discrepancy in glider found for civl_id 73646:
old glider:  Advance Omega ULS
new glider:  Advance Omega X-Alps 4

Discrepancy in glider found for civl_id 90734:
old glider:  Niviuk Klimber 2 P
new glider:  Niviuk Klimber 2

Discrepancy in glider found for civl_id 64916:
old glider:  Ozone Zeolite GT 2
new glider:  PHI TENOR

Discrepancy in glider found for civl_id 55154:
old glider:  Ozone Zeolite 2 GT
new glider:  Ozone Zeolite GT

Discrepancy in glider found for civl_id 73663:
old glider:  Niviuk Artik 7 P
n

Cleaning the gliders In terms writting style

In [ ]:
def normalize_glider_name(glider):
    if pd.isna(glider):
        return ""

    glider = glider.lower().strip()

    # Common replacements
    replacements = {
        "six": "6",
        "swift6": "swift 6",
        "7p": "7 p",
        "6p": "6 p",
        "3p": "3 p",
        "volt4": "volt 4",
        "volt5": "volt 5",
        "enzo3": "enzo 3",
        "oxp2": "oxa 2",
        "omegauls": "omega uls",
        "air design": "airdesign",
    }
    for old, new in replacements.items():
        glider = glider.replace(old, new)

    # Special P-suffix handling
    def fix_p_suffix(model):
        if f"{model} " in glider or f"{model}" in glider:
            if "p" not in glider:
                return glider.replace(model, f"{model} p")
        return glider

    glider = fix_p_suffix("artik 7")
    glider = fix_p_suffix("klimber 3")
    glider = fix_p_suffix("klimber 2")
    glider = fix_p_suffix("ikuma 3")

    # Add manufacturer if missing
    manufacturers = {
        "ozone": ["zeolite", "swift", "lygth", "alpina"],
        "advance": ["omega", "sigma", "iota", "theta"],
        "niviuk": ["artik", "klimber", "hiko", "ikuma"],
        "phi": ["allegro", "scala", "beat"],
        "gin": ["explorer"],
        "airdesign": ["hero", "soar", "volt"],
        "skywalk": ["arak", "sage"],
        "nova": ["mentor", "xenon", "vortex", "codex"],
    }
    for brand, keywords in manufacturers.items():
        if any(k in glider for k in keywords) and brand not in glider:
            glider = f"{brand} {glider}"
            break

    # Title-case + brand-specific fixes
    glider = glider.title()
    brand_fixes = {
        "Airdesign": "AirDesign",
        "Phi": "PHI",
        "Supair": "SupAir",
        "Uls": "ULS",
        "Dls": "DLS",
        "Rs": "RS",
        "Gt": "GT",
    }
    for wrong, correct in brand_fixes.items():
        glider = glider.replace(wrong, correct)

    return glider


# Apply to json_data
for athlete in json_data.keys():
    json_data[athlete]["glider"] = normalize_glider_name(json_data[athlete]["glider"])



Check if there is Athletes with the same Name but different CIVIL ID

In [ ]:
name_map = defaultdict(list)

for civl_id, data in json_data.items():
    name = data.get('name', '').strip().lower()
    first_name = data.get('first_name', '').strip().lower()

    # Build both "name-first_name" and "first_name-name" as possible keys
    key1 = (name, first_name)
    key2 = (first_name, name)

    name_map[key1].append(civl_id)
    if key1 != key2:
        name_map[key2].append(civl_id)

# Filter and print duplicates
printed = set()
for key, ids in name_map.items():
    unique_ids = set(ids)
    if len(unique_ids) > 1 and key not in printed:
        printed.add(key)
        print(f"Possible duplicate: {key[0].title()} {key[1].title()} found with CIVL IDs: {', '.join(sorted(unique_ids))}")


Check that all Genders are F or M


In [ ]:
def normalize_gender(gender):
    gender = gender.strip().upper()

    if len(gender) > 1:
        print(f"Invalid Gender length. Gender: {gender}")

    if gender == "W":   # normalize "W" to "F"
        gender = "F"

    return gender


# Apply to json_data
for athlete in json_data.keys():
    json_data[athlete]["gender"] = normalize_gender(json_data[athlete]["gender"])

Check that Nationalities are a 3 digit word and same countries are represented the same

In [ ]:
def normalize_nationality(nationality):
    nationality = nationality.strip().upper()

    replacements = {
        "CH": "SUI",
        "IT": "ITA",
        "UK": "GBR",
        "FR": "FRA",
        "F": "FRA",
        "NZ": "NZL",
        "D": "DEU",
        "AT": "AUT",
        "DK": "DNK",
        "UY": "URY",
        "DE": "DEU",
    }

    if nationality in replacements:
        nationality = replacements[nationality]

    if len(nationality) != 3:
        print(f"Invalid Nationality length. Nationality: {nationality}")

    return nationality

#apply to JSON data
for athlete in json_data.keys():
    json_data[athlete]["nat"] = normalize_nationality(json_data[athlete]["nat"])

Invalid Nationality length. Nationality: -
Invalid Nationality length. Nationality: -


Save Data to JSON


In [ ]:
# Save to JSON files
with open(json_path, "w") as f:
    json.dump(json_data, f, indent=4)

with open(competition_path, "w") as f:
    json.dump(competition_json, f, indent=4)

### Generate PDFs for Publication

Functions needed for PDF generation

In [ ]:
def add_data_to_lists(overall_rows: list, gender_rows: list, athlete_data: dict, competitions: list, civil_id):
    row = {
        "name": f"{athlete_data['first_name']} {athlete_data['name']}",
        }
    for comp in competitions:
        try:
           row[comp] = athlete_data["competitions"][comp]["points"]
        except KeyError:
           row[comp] = ""
    row["total_points"] = athlete_data["total_points"]
    row["civil_id"] = civil_id
    row["gender"] = athlete_data["gender"]
    row["nat"] = athlete_data["nat"]
    row["glider"] = athlete_data["glider"]

    # add the data row to the lists
    overall_rows.append(row), gender_rows.append(row)

    return overall_rows, gender_rows


def generate_pandas_data_frames(json_data: dict, competitions: list):
    rows_female = []
    rows_male = []
    rows_overall = []
    for civl_id in json_data.keys():
        if json_data[civl_id]['gender'] == 'F':
            rows_overall, rows_female = add_data_to_lists(rows_overall, rows_female, json_data[civl_id], all_competitions, civl_id)
        else:
            rows_overall, rows_male = add_data_to_lists(rows_overall, rows_male, json_data[civl_id], all_competitions, civl_id)

    df_female = pd.DataFrame(rows_female)
    df_male = pd.DataFrame(rows_male)
    df_overall = pd.DataFrame(rows_overall)

    return df_female, df_male, df_overall


def extract_competition_name_list(competition_json: dict):
    names = []
    for comp in competition_json.keys():
        names.append(competition_json[comp]["title"])
    return names



class RotatedHeader(Flowable):
    def __init__(self, text, width=40, height=60, fontSize=10):
        super().__init__()
        self.text = text
        self.width = width
        self.height = height
        self.fontSize = fontSize

    def draw(self):
        self.canv.saveState()
        self.canv.setFont("Helvetica-Bold", self.fontSize)

        # Move origin to the bottom center of the cell
        self.canv.translate(0, 0)

        # Rotate around the new origin
        self.canv.rotate(90)

        # Draw string so it's centered and touches bottom
        text_width = self.canv.stringWidth(self.text, "Helvetica", self.fontSize)
        self.canv.drawString(5 , -24, self.text) # the first number aligns up/down, smaller -> down, secon number moves left/rigth, smaller -> right, but they also influence each other

        self.canv.restoreState()

    def wrap(self, availWidth, availHeight):
        return self.width, self.height




def generate_single_pdf(json_data: dict, competition_json: dict, df: pd.DataFrame, titel: str, path: str):
    doc = SimpleDocTemplate(str(path), pagesize=landscape(A4), leftMargin=20, rightMargin=20, topMargin=30, bottomMargin=20)

    styles = getSampleStyleSheet()
    story = []

    # Load and resize image with preserved aspect ratio
    img_reader = ImageReader(logo_path)
    orig_width, orig_height = img_reader.getSize()
    target_width = 100
    aspect_ratio = orig_height / orig_width
    target_height = target_width * aspect_ratio
    logo = Image(logo_path, width=target_width, height=target_height)

    # Create title paragraph
    title_para = Paragraph(f"<b>{titel}</b>", styles["Title"])

    # Create a 1-row, 2-column table with logo and title
    title_table = Table([[title_para, logo]], colWidths=[400, target_width + 10])  # Adjust second colWidth as needed
    title_table.setStyle(TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (1, 0), (1, 0), "LEFT"),  # Align title left if desired
        ("LEFTPADDING", (0, 0), (-1, -1), 0),
        ("RIGHTPADDING", (0, 0), (-1, -1), 0),
        ("TOPPADDING", (0, 0), (-1, -1), 0),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 0),
    ]))

    # Add to story
    story.append(title_table)
    story.append(Spacer(1, 12))

    # add infos, so called "Kleingedrucktes"
    # Create a custom small grey text style
    info_style = ParagraphStyle(
        name="InfoStyle",
        parent=styles["Normal"],
        fontSize=8,
        textColor=grey,
        spaceBefore=2,
        spaceAfter=2,
    )

    story.append(Paragraph(info, info_style))

    # Prepare table header
    competitions = extract_competition_name_list(competition_json)
    raw_header = ["Rank", "Civil ID", "Name", "Gender", "Nationality", "Glider"] + competitions + ["Total Points"]
    header = []

    for i, col in enumerate(raw_header):
         header.append(RotatedHeader(col))

    # Prepare table rows
    table_data = [header]
    highlight_cells = []
    previous_points = 0
    pervious_rank = 0
    for rank, (idx, row) in enumerate(df.iterrows(), 1):
        if row["total_points"] == previous_points:
            athletes_true_rank = pervious_rank
        else:
            athletes_true_rank = rank
            previous_points = row["total_points"]
            pervious_rank = rank
        row_data = [str(athletes_true_rank), row["civil_id"], row["name"], row["gender"], row["nat"], row["glider"]]

        for offset, comp_key in enumerate(competition_json.keys()):
            row_data.append(str(row[comp_key]))

            try:
              # If this comp counts for the athlete, store (row_index, col_index) for highlighting
              if json_data[row["civil_id"]]["competitions"][comp_key]["counts"]:
                  table_row_idx = len(table_data)  # current row index in table
                  table_col_idx = 6 + offset     # offset by Rank and Name, gender, nat, glider columns
                  highlight_cells.append((table_row_idx, table_col_idx))
            except KeyError:
              continue

        row_data.append(str(row.get("total_points", "")))
        table_data.append(row_data)

    # Build the table
    col_widths = [20, 40, 135, 20, 30, 140] + [30] * len(competitions) + [35]
    row_heights = [155] + [None] * (len(table_data) - 1)
    table = Table(table_data, colWidths=col_widths, rowHeights=row_heights, repeatRows=1)

    style = TableStyle([
        # Basic grid
        ("GRID", (0, 0), (-1, -1), 0.5, colors.black),

        # Header styling
        ("ALIGN", (0, 0), (-1, 0), "CENTER"),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),

        # Body alignment

      ("ALIGN", (0, 1), (0, -1), "CENTER"),  # Rank
      ("ALIGN", (1, 1), (1, -1), "LEFT"),    # Civil id
      ("ALIGN", (2, 1), (2, -1), "LEFT"),    # Name
      ("ALIGN", (3, 1), (4, -1), "CENTER"),  # Gender and Nationality
      ("ALIGN", (5, 1), (5, -1), "LEFT"),    # Glider
      ("ALIGN", (6, 1), (-1, -1), "CENTER"), # Competitions + Total Points


        # Bold outer border
        ("BOX", (0, 0), (-1, -1), 1.5, colors.black),

        # Bold line after header row
        ("LINEBELOW", (0, 0), (-1, 0), 1.5, colors.black),

        # Bold vertical line after sixth column (Glider)
        ("LINEAFTER", (5, 0), (5, -1), 1.5, colors.black),

        # Bold vertical line before last column (Total Points)
        ("LINEBEFORE", (-1, 0), (-1, -1), 1.5, colors.black),
    ])

    highlight_color = colors.Color(red=172/255, green=220/255, blue=149/255, alpha = 0.6)


    # Light grey background for alternating rows (excluding header)
    for i in range(1, len(table_data)):
        if i % 2 == 0:  # Even-numbered row (index starts at 0)
            style.add("BACKGROUND", (0, i), (-1, i), colors.whitesmoke)

    for r, c in highlight_cells:
        style.add("BACKGROUND", (c, r), (c, r), highlight_color)

    table.setStyle(style)

    story.append(table)
    doc.build(story)


Run the PDF generation

In [ ]:
all_competitions = competition_json.keys()

df_female, df_male, df_overall = generate_pandas_data_frames(json_data, all_competitions)

# Filter out rows where total_points is 0
df_female = df_female[df_female["total_points"] != 0]
df_male = df_male[df_male["total_points"] != 0]
df_overall = df_overall[df_overall["total_points"] != 0]


# Sort by total points descending
df_female.sort_values("total_points", ascending=False, inplace=True)
df_male.sort_values("total_points", ascending=False, inplace=True)
df_overall.sort_values("total_points", ascending=False, inplace=True)

# generate the PDFs
generate_single_pdf(json_data, competition_json, df_female, women_title, files_path / "swisscup_hf_2025_female.pdf")
generate_single_pdf(json_data, competition_json, df_male, men_title, files_path / "swisscup_hf_2025_male.pdf")
generate_single_pdf(json_data, competition_json, df_overall, overall_title, files_path / "swisscup_hf_2025_overall.pdf")

Delete all Competions but keep the rest of the data. Only do this if you need to reset...

In [8]:
''' for civil_id in json_data:
  json_data[civil_id]["competitions"] = {}
  json_data[civil_id]["total_points"] = 0 '''

In [9]:
# Save to JSON file
with open(f"{json_path}", "w") as f:
    json.dump(json_data, f, indent=4)